# Concept
### เป้าหมาย : 
นับจำนวนคนที่เดินเข้าประตูได้
### วิธีการ : 
สังเกตุจากวีดีโอจะเห็นได้ว่าทิศทางของคนเดินเข้าประตูจะมีลักษณะเป็นทิศทางที่ตรงกันข้ามกับคนที่เดินออกจากประตู ดังนั้นผมจึงเลือกที่จะใช้วิธีการเป็น สร้าง ROI (Region of Interest) ไว้ตรงประตู และdetect ทิศทางของคนที่เดินผ่าน ROI นี้ว่ามีทิศทางที่เดินเข้าไปในประตูหรือไม่ แต่ว่าเนื่องจาก ROI เป็น Plane 2 มิติ ทำให้ทิศทางของคนที่เดินเข้าประตู หรือเดินออกทางซ้ายของประตูมี vector ที่เป็นในทิศทางเดียวกันทำให้เกิดข้อผิดพลาดในการนับได้ ดังนั้นผมจึงสร้างเส้นแนวตั้ง 1 เส้นไว้ทางข้างๆประตูฝั่งซ้าย เพื่อเช็คดูว่าคนที่เดินในทิศที่เข้าประตูนี้ เข้าประตูจริงๆ หรือเดินไปทางห้องน้ำครับ

![detection ROI](./img/detection_roi.png)

Logic:
- คนเดินเข้าห้องผ่านพื้นที่ ROI แต่ไม่ผ่านเส้น cancel line -> Pending state -> การ detect คนๆนั้นหายไปจาก ROI -> เข้าห้องเรียบร้อย -> นับ 1
- คนเดินในทิศทางที่ไปทางห้องน้ำ ผ่านเส้น cancel line -> นับ 0

# Import Library

In [1]:
import sys
import cv2
from ultralytics import YOLO
from collections import defaultdict ,deque

# Config

ROI -> กรอบประตู (x1,y1,x2,y2)

OUTSIDE_ZONE -> เส้นแนวตั้งบน ROI ใช้เป็นเกณฑ์ในการนับคนเข้าห้อง

TRAIL_LEN -> เก็บตำแหน่ง frame ย้อนหลัง เพื่อนำไปคำนวณ vector

MIN_DX -> displacement ขั้นต่ำ (px) ถึงจะถือว่า "กำลังเดิน"

COOLDOWN_FRAMES -> กัน track เดิมถูกนับซ้ำเร็วเกินไป

TIMEOUT_FRAMES -> ถ้า track หายไป 90 frame → ถือว่าเข้าอาคารแล้ว

PROCESS_EVERY_N -> run inference ทุก 2 frame (เพิ่มความเร็ว)


In [2]:
PERSON_CLASS_ID = 0
MODEL_NAME      = "yolov8n.pt"
BOX_COLOR       = (0, 255, 80)
TEXT_COLOR      = (255, 255, 255)
FONT            = cv2.FONT_HERSHEY_SIMPLEX

ROI             = (520, 80, 920, 780)
OUTSIDE_ZONE_X  = 720
CANCEL_LINE_X   = 350

TRAIL_LEN       = 15
MIN_DX          = 25
COOLDOWN_FRAMES = 90
TIMEOUT_FRAMES  = 90
PROCESS_EVERY_N = 2

# Create Function

In [3]:
# หาจุดกึ่งกลางของกล่องที่ detect ได้จาก Yolo model
def get_centroid(box):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    return (x1 + x2) // 2, (y1 + y2) // 2

In [4]:
# วาด bounding box + label สำหรับคนที่ตรวจพบ
def draw_people(frame, box):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    conf = float(box.conf[0])
    cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)
    label = f"person {conf:.2f}"
    (tw, th), _ = cv2.getTextSize(label, FONT, 0.55, 1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), BOX_COLOR, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4), FONT, 0.55, TEXT_COLOR, 1, cv2.LINE_AA)

In [5]:
# วาดเส้นและกรอบ reference บนทุกๆ frame  ได้แก่ กรอบ DOOR ROI (สีเหลือง), เส้น cancel (สีแดง) และเส้น outside zone (สีน้ำเงินอ่อน)
def draw_overlay(frame, count_in):
    rx1, y1, rx2, y2 = ROI

    # Cancel line (สีแดง)
    cv2.line(frame, (CANCEL_LINE_X, y1), (CANCEL_LINE_X, y2), (0, 0, 255), 2)
    cv2.putText(frame, "CANCEL", (CANCEL_LINE_X + 4, y1 + 20), FONT, 0.55, (0, 0, 255), 1)

    # Outside zone (สีน้ำเงินอ่อน)
    cv2.line(frame, (OUTSIDE_ZONE_X, y1), (OUTSIDE_ZONE_X, y2), (100, 100, 255), 2)
    cv2.putText(frame, "outside", (OUTSIDE_ZONE_X + 4, y1 + 20), FONT, 0.5, (100, 100, 255), 1)

    # ROI box
    cv2.rectangle(frame, (rx1, y1), (rx2, y2), (0, 255, 255), 2)
    cv2.putText(frame, "DOOR ROI", (rx1 + 6, y1 + 28), FONT, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"IN: {count_in}", (30, 60), FONT, 1.4, (0, 255, 100), 2, cv2.LINE_AA)

![Run Function](./img/RunFunction1.jpg)

In [6]:
def run(args):
    print(f"[INFO] Loading model: {MODEL_NAME}")
    model = YOLO(MODEL_NAME)

    source = args.input if args.input != "0" else 0
    cap    = cv2.VideoCapture(source)
    if not cap.isOpened():
        sys.exit(f"[ERROR] Cannot open: {source}")

    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = args.output or "result.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    writer   = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    print(f"[INFO] Source      : {source}  ({w}x{h} @ {fps:.1f} fps, ~{total_frames} frames)")
    print(f"[INFO] ROI         : {ROI}")
    print(f"[INFO] OUTSIDE_ZONE: x={OUTSIDE_ZONE_X}")
    print(f"[INFO] CANCEL_LINE : x={CANCEL_LINE_X}")
    print("[INFO] Running — press Q in preview window to stop early\n")

    frame_idx         = 0
    count_in          = 0
    track_trails      = defaultdict(lambda: deque(maxlen=TRAIL_LEN))
    track_last_count  = {}
    track_last_seen   = {}
    track_was_outside = {}
    track_pending_in  = {}

    x1r, y1r, x2r, y2r = ROI
    cached_draws = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % PROCESS_EVERY_N == 0:
            results = model.track(frame, classes=[PERSON_CLASS_ID],
                                  conf=args.conf, persist=True, verbose=False)
            cached_draws = []

            if results[0].boxes.id is not None:
                for box, track_id in zip(results[0].boxes,
                                         results[0].boxes.id.int().tolist()):
                    cx, cy = get_centroid(box)
                    track_last_seen[track_id] = frame_idx

                    if cx > OUTSIDE_ZONE_X:
                        track_was_outside[track_id] = True

                    trail = track_trails[track_id]
                    trail.append((cx, cy))

                    if track_pending_in.get(track_id, False) and cx < CANCEL_LINE_X:
                        track_pending_in[track_id] = False
                        print(f"  [TRACK {track_id}] ❌ ยกเลิก (ข้าม cancel line x={cx})")

                    if track_pending_in.get(track_id, False) and cx > x2r:
                        track_pending_in[track_id] = False
                        print(f"  [TRACK {track_id}] ❌ ยกเลิก (หันกลับ x={cx})")

                    in_door = x1r <= cx <= x2r and y1r <= cy <= y2r
                    if in_door and len(trail) >= max(3, TRAIL_LEN // 2):
                        dx         = trail[-1][0] - trail[0][0]
                        last_count = track_last_count.get(track_id, -COOLDOWN_FRAMES)
                        if dx < -MIN_DX and (frame_idx - last_count) > COOLDOWN_FRAMES \
                                        and track_was_outside.get(track_id, False) \
                                        and not track_pending_in.get(track_id, False):
                            track_pending_in[track_id]  = True
                            track_last_count[track_id]  = frame_idx
                            track_was_outside[track_id] = False
                            print(f"  [TRACK {track_id}] 🔄 pending IN  dx={dx:+d}")

                    pending = track_pending_in.get(track_id, False)
                    was_out = track_was_outside.get(track_id, False)
                    dx_disp = trail[-1][0] - trail[0][0] if len(trail) > 1 else 0
                    flag    = "P" if pending else ("O" if was_out else ".")
                    color   = (0, 255, 255) if pending else (0, 255, 100)

                    x1b, y1b, x2b, y2b = map(int, box.xyxy[0])
                    cached_draws.append({
                        'box':           (x1b, y1b, x2b, y2b, float(box.conf[0])),
                        'trail':         list(trail),
                        'label':         (f"ID{track_id}[{flag}] dx={dx_disp:+d}", (cx + 8, cy - 8), color),
                        'centroid':      (cx, cy),
                        'bottom_center': (cx, y2b),
                    })

            for tid in list(track_trails.keys()):
                if frame_idx - track_last_seen.get(tid, 0) > TIMEOUT_FRAMES:
                    if track_pending_in.get(tid, False):
                        count_in += 1
                        print(f"  [TRACK {tid}] ✅ เข้า! (หายเข้าอาคาร)  count_in={count_in}")
                    del track_trails[tid]
                    track_last_count.pop(tid, None)
                    track_was_outside.pop(tid, None)
                    track_pending_in.pop(tid, None)

        for d in cached_draws:
            x1b, y1b, x2b, y2b, conf = d['box']
            cv2.rectangle(frame, (x1b, y1b), (x2b, y2b), BOX_COLOR, 2)
            lbl = f"person {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(lbl, FONT, 0.55, 1)
            cv2.rectangle(frame, (x1b, y1b - th - 6), (x1b + tw + 4, y1b), BOX_COLOR, -1)
            cv2.putText(frame, lbl, (x1b + 2, y1b - 4), FONT, 0.55, TEXT_COLOR, 1, cv2.LINE_AA)

            pts = d['trail']
            for i in range(1, len(pts)):
                cv2.line(frame, pts[i - 1], pts[i], (255, 165, 0), 1)

            text, pos, color = d['label']
            cv2.putText(frame, text, pos, FONT, 0.42, color, 1)

            cv2.circle(frame, d['centroid'], 5, (0, 255, 255), -1)
            cv2.circle(frame, d['bottom_center'], 5, (255, 0, 255), -1)

        draw_overlay(frame, count_in)
        writer.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            pct = f"{frame_idx/total_frames*100:.1f}%" if total_frames else f"{frame_idx} frames"
            print(f"[INFO] Progress: {pct}  |  IN={count_in}")

        if args.show:
            cv2.imshow("People Counter — Q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                print("[INFO] Stopped by user.")
                break

    for tid, pending in track_pending_in.items():
        if pending:
            count_in += 1
            print(f"  [TRACK {tid}] ✅ เข้า! (end-of-video)  count_in={count_in}")

    cap.release()
    writer.release()
    if args.show:
        for _ in range(5):
            cv2.destroyAllWindows()
            cv2.waitKey(1)

    print(f"\n[DONE] Processed : {frame_idx} frames")
    print(f"       Total IN  : {count_in}")
    print(f"       Output    : {out_path}")

In [7]:
class Args:
    input   = "entrance.mov"   # ← เปลี่ยนชื่อไฟล์ตรงนี้
    output  = "result.mp4"
    conf    = 0.35
    show    = True
    summary = True

run(Args())

[INFO] Loading model: yolov8n.pt
[INFO] Source      : entrance.mov  (1920x1080 @ 29.9 fps, ~2556 frames)
[INFO] ROI         : (520, 80, 920, 780)
[INFO] OUTSIDE_ZONE: x=720
[INFO] CANCEL_LINE : x=350
[INFO] Running — press Q in preview window to stop early

  [TRACK 3] 🔄 pending IN  dx=-133
[INFO] Progress: 3.9%  |  IN=0
  [TRACK 3] ❌ ยกเลิก (ข้าม cancel line x=337)
  [TRACK 13] 🔄 pending IN  dx=-426
[INFO] Progress: 7.8%  |  IN=0
[INFO] Progress: 11.7%  |  IN=0
  [TRACK 13] ✅ เข้า! (หายเข้าอาคาร)  count_in=1
[INFO] Progress: 15.6%  |  IN=1
  [TRACK 32] 🔄 pending IN  dx=-134
  [TRACK 32] ❌ ยกเลิก (ข้าม cancel line x=329)
[INFO] Progress: 19.6%  |  IN=1
  [TRACK 54] 🔄 pending IN  dx=-208
  [TRACK 54] ❌ ยกเลิก (ข้าม cancel line x=306)
  [TRACK 49] 🔄 pending IN  dx=-43
[INFO] Progress: 23.5%  |  IN=1
[INFO] Progress: 27.4%  |  IN=1
  [TRACK 50] 🔄 pending IN  dx=-42
  [TRACK 49] ✅ เข้า! (หายเข้าอาคาร)  count_in=2
[INFO] Progress: 31.3%  |  IN=2
  [TRACK 68] 🔄 pending IN  dx=-114
  [TRACK 5